# 10 — Apply the xformers Fix & Certify Inits Train-Ready

**Confirmed root cause (nb08+nb09):** the init weights are byte-perfect; the forward NaN was
NeoBERT using xformers' fused SwiGLU under an incompatible build (torch 2.11/cu128). With
xformers absent, the pure-torch SwiGLU gives a finite forward (loss 6.66).

**Fix:** `salt3_common._patch_model_py_xformers` now rewrites each saved init's `model.py` to
use the pure-torch SwiGLU **unconditionally** (no xformers import at all), so a present-but-
broken xformers can no longer be picked up. This notebook re-patches existing init artifacts
on Drive and certifies forward is finite WITHOUT xformers.

Run order: make sure the updated `salt3_common.py` is synced to `SALT3/code/` first.


In [ ]:
%%capture
# Deliberately NO xformers — proving the pipeline no longer needs it.
!pip uninstall -y xformers
!pip install -U transformers safetensors huggingface_hub sentencepiece accelerate


In [ ]:
import sys, math
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoModelForMaskedLM, AutoTokenizer

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)
PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
sys.path.insert(0, str(PROJECT_ROOT / 'code')); sys.path.insert(0, '/content')
import importlib, salt3_common as sc
importlib.reload(sc)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

import transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
try:
    import xformers; print('xformers STILL present:', xformers.__version__, '(fix makes this irrelevant)')
except Exception:
    print('xformers absent (intended)')

# Every init artifact that has a model/ dir
INIT_ROOT = PROJECT_ROOT / 'init'
INITS = [p.parent for p in INIT_ROOT.glob('*/model') if (p / 'model.py').exists()]
print('init artifacts found:', [p.name for p in INITS])


## 1. Re-patch each init's model.py to the pure-torch SwiGLU
`_patch_model_py_xformers` is idempotent and upgrades the old try/except patch in place.


In [ ]:
for d in INITS:
    mp = d / 'model' / 'model.py'
    before = mp.read_text(encoding='utf-8')
    sc._patch_model_py_xformers(mp)
    after = mp.read_text(encoding='utf-8')
    has_marker = sc._PURE_SWIGLU_MARKER in after
    imports_xf = any(l.strip().startswith(('import xformers', 'from xformers'))
                     for l in after.splitlines())
    print(f'{d.name:42s} patched={before != after} pure_marker={has_marker} imports_xformers={imports_xf}')
# Clear the cached remote module so the re-patched model.py is the one that loads.
import shutil, glob, os
for c in glob.glob('/root/.cache/huggingface/modules/transformers_modules/*'):
    pass  # cache is keyed by content hash; a changed model.py gets a new hash automatically


## 2. Certify each init: loads + forward FINITE without xformers


In [ ]:
def make_batch(tok):
    sents = ['Việt Nam là một quốc gia ở Đông Nam Á.',
             'Hôm nay thời tiết rất đẹp và trời trong xanh.',
             'Kinh tế Việt Nam tăng trưởng trong năm qua.',
             'Trẻ em cần được tiêm phòng đầy đủ để tránh bệnh.']
    return tok(sents, padding=True, truncation=True, max_length=32, return_tensors='pt').to(DEVICE)

results = []
for d in INITS:
    md_dir = d / 'model'
    try:
        model = AutoModelForMaskedLM.from_pretrained(md_dir, trust_remote_code=True).to(DEVICE).eval()
        tok = AutoTokenizer.from_pretrained(md_dir, trust_remote_code=True)
        ffn_t = type(model.model.transformer_encoder[0].ffn).__name__
        enc = make_batch(tok); ids = enc['input_ids']
        torch.manual_seed(0)
        pm = torch.full(ids.shape, 0.2)
        for sid in set(tok.all_special_ids): pm[ids.cpu() == sid] = 0
        msk = torch.bernoulli(pm).bool().to(DEVICE)
        lab = torch.full_like(ids, -100); lab[msk] = ids[msk]
        mids = ids.clone(); mids[msk] = tok.mask_token_id
        with torch.no_grad():
            lg = model(input_ids=mids, attention_mask=enc['attention_mask']).logits
        fin = bool(torch.isfinite(lg).all())
        loss = F.cross_entropy(lg.reshape(-1, lg.size(-1)).float(), lab.reshape(-1), ignore_index=-100).item()
        results.append((d.name, ffn_t, fin, loss))
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception as e:
        results.append((d.name, 'LOAD-FAIL', False, repr(e)[:60]))

print('\n' + '=' * 78)
print(f'{"init":42s} {"ffn":10s} {"finite":7s} loss')
for name, ffn, fin, loss in results:
    print(f'{name:42s} {ffn:10s} {str(fin):7s} {loss}')
print('=' * 78)
allok = all(fin for _, _, fin, _ in results)
print('✅ ALL INITS FORWARD FINITE WITHOUT XFORMERS — pipeline unblocked, safe to train (nb02).'
      if allok else '❌ some init still fails — inspect rows above.')
print('=' * 78)


## 3. Next
- Training (nb02): with these re-patched inits, CPT runs without xformers. Remove any
  `pip install xformers` from nb02 (it can only reintroduce the broken fused kernel).
- The donor-quality question (nb07) is SEPARATE and now unblocked — re-run with mean-centering
  on a working forward.
